# XACLE 2027 — Experiment D02: MOE8 Gate Entropy Diagnostics

A read-only diagnostic of the frozen Full8 gate on official train, validation, and public test. It consumes the precomputed checkpoint, run manifest, config, and Full8 stores from the moescore_full8_artifacts dataset. It does not extract expert features, assemble stores, or train models.

We summarize routing weights and entropy by split, score bin, and generator where verified metadata permits; then compare learned routing with uniform and hard top-1 controls on matched samples. Public-test labels, if available from the official metadata, are evaluation-only. The private evaluation dataset is not attached or accessed.

D02 is descriptive and cannot establish that gate entropy causes score compression or prediction errors. Results may motivate a separate G01 entropy ablation.

In [ ]:
# 2. Runtime configuration
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, shlex, subprocess, sys

EXPERIMENT_ID = "D02-MOE8"
PROFILE = "full_8"
USE_PRECOMPUTED_FULL8_STORE = True
RUN_TRAIN_ANALYSIS = True
RUN_VALIDATION_ANALYSIS = True
RUN_PUBLIC_TEST_ANALYSIS = True
RUN_FEATURE_EXTRACTION = False
RUN_FULL7_ASSEMBLY = False
RUN_FULL8_ASSEMBLY = False
TRAIN_GATE = False
TRAIN_EXPERTS = False
SKIP_CROSS_SPLIT_AUDIO_OVERLAP_CHECK = True
BATCH_SIZE = 128
OUTPUT_ROOT = Path("/kaggle/working/d02_moe8")
assert USE_PRECOMPUTED_FULL8_STORE and not any(
    (RUN_FEATURE_EXTRACTION, RUN_FULL7_ASSEMBLY, RUN_FULL8_ASSEMBLY, TRAIN_GATE, TRAIN_EXPERTS)
)
assert SKIP_CROSS_SPLIT_AUDIO_OVERLAP_CHECK
for folder in ("diagnostics", "figures", "reports", "logs"):
    (OUTPUT_ROOT / folder).mkdir(parents=True, exist_ok=True)
EXECUTION_LOG = OUTPUT_ROOT / "logs/execution.log"
EXECUTION_LOG.write_text("", encoding="utf-8")
STAGES = {}

def log_event(event, **details):
    row = {"time_utc": datetime.now(timezone.utc).isoformat(), "event": event, **details}
    with EXECUTION_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, sort_keys=True, default=str) + "\n")
    print(event, details)

def mark_stage(name, status, **details):
    STAGES[name] = {"status": status, **details}
    log_event("stage_" + status.lower(), stage=name, **details)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True,
                                     allow_nan=False, default=str) + "\n", encoding="utf-8")

def run_subprocess(command, *, cwd=None, log_path=None):
    command = [str(part) for part in command]
    print("$ " + shlex.join(command), flush=True)
    try:
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True, check=False)
    except OSError as exc:
        message = f"Command could not start: {shlex.join(command)}: {type(exc).__name__}: {exc}"
        print(message, file=sys.stderr, flush=True)
        if log_path is not None: Path(log_path).write_text(message + "\n", encoding="utf-8")
        raise
    stdout, stderr = result.stdout or "", result.stderr or ""
    if stdout: print(stdout, end="" if stdout.endswith("\n") else "\n", flush=True)
    if stderr: print(stderr, file=sys.stderr, end="" if stderr.endswith("\n") else "\n", flush=True)
    if log_path is not None:
        Path(log_path).write_text("STDOUT\n" + stdout + "\nSTDERR\n" + stderr, encoding="utf-8")
    if result.returncode:
        message = f"Command failed with exit code {result.returncode}: {shlex.join(command)}"
        print(message, file=sys.stderr, flush=True)
        raise subprocess.CalledProcessError(result.returncode, command, output=stdout, stderr=stderr)
    if not stdout and not stderr: print("(command completed without output)", flush=True)
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""): digest.update(block)
    return digest.hexdigest()

mark_stage("configuration", "PASS", training=False, splits=["train", "validation", "test"],
           cross_split_audio_overlap="SKIPPED")

In [ ]:
# 3. Pin checkpoint-compatible source and D02 analysis module
import importlib.util
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
MOE8_SOURCE_BRANCH = "fix/full8-kaggle-save-run-all"
MOE8_SOURCE_COMMIT = "68124d0e57005d29149f659780ab749d14e09532"
D02_HELPER_BRANCH = "research/xacle2027-d02-moe8-gate-diagnostics"
D02_HELPER_COMMIT = "e337d5d34f0e3fd22c834be98fb62254b951e698"
MOE8_SOURCE_ROOT = Path("/kaggle/working/moescore_moe8_source")
D02_SOURCE_ROOT = Path("/kaggle/working/moescore_d02_source")

def checkout_pin(path, branch, commit):
    if len(commit) != 40 or any(c not in "0123456789abcdef" for c in commit):
        raise ValueError(f"Invalid source commit pin: {commit!r}")
    if path.exists():
        if not (path / ".git").exists(): raise RuntimeError(f"Not a Git checkout: {path}")
        origin = run_subprocess(["git", "-C", path, "remote", "get-url", "origin"]).stdout.strip()
        if origin.rstrip("/") != REPO_URL.rstrip("/"): raise RuntimeError(f"Unexpected Git origin: {origin}")
    else:
        run_subprocess(["git", "clone", "--filter=blob:none", "--branch", branch,
                        "--single-branch", REPO_URL, path])
    run_subprocess(["git", "-C", path, "fetch", "--depth=1", "origin", commit])
    run_subprocess(["git", "-C", path, "checkout", "--detach", commit])
    actual = run_subprocess(["git", "-C", path, "rev-parse", "HEAD"]).stdout.strip()
    if actual != commit: raise RuntimeError(f"Expected {commit}; checked out {actual}")
    return actual

MOE8_SOURCE_SHA = checkout_pin(MOE8_SOURCE_ROOT, MOE8_SOURCE_BRANCH, MOE8_SOURCE_COMMIT)
D02_SOURCE_SHA = checkout_pin(D02_SOURCE_ROOT, D02_HELPER_BRANCH, D02_HELPER_COMMIT)
missing = [n for n in ("torch", "numpy", "pandas", "scipy", "matplotlib", "yaml")
           if importlib.util.find_spec(n) is None]
if missing: raise RuntimeError(f"Missing runtime packages: {missing}")
os.chdir(MOE8_SOURCE_ROOT)
sys.path.insert(0, str(MOE8_SOURCE_ROOT))
import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from model.expert_adapters.registry import profile_contract
from model.moe_gating_core import GatingNetwork
from training.checkpoint_manager import MoeCheckpointManager
from data.moe_feature_store import validate_feature_store
from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features

helper_path = D02_SOURCE_ROOT / "research/xacle27/analysis/d02_gate_diagnostics.py"
spec = importlib.util.spec_from_file_location("d02_gate_diagnostics_pinned", helper_path)
D02 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = D02
spec.loader.exec_module(D02)
EXPERT_ORDER, EXPERT_DIMS, FEATURE_WIDTH = profile_contract(PROFILE)
if FEATURE_WIDTH != 33032 or len(EXPERT_ORDER) != 8:
    raise RuntimeError("Pinned Full8 profile contract has incompatible dimensions")
mark_stage("source_and_dependencies", "PASS", moe8_source=MOE8_SOURCE_SHA,
           d02_analysis_source=D02_SOURCE_SHA, expert_order=list(EXPERT_ORDER),
           feature_width=FEATURE_WIDTH, execution_device="cpu")

In [ ]:
# 4. Discover mounted inputs without accessing private evaluation data
KAGGLE_INPUT = Path("/kaggle/input")
if not KAGGLE_INPUT.is_dir(): raise FileNotFoundError("/kaggle/input is unavailable")
DATASET_REFS = {"full8_artifacts": "qnv8071/moescore-full8-artifacts",
                "official_metadata_optional": "qnv8071/xacle-2027-data"}
DATASET_VERSIONS_AT_UPLOAD = {"qnv8071/moescore-full8-artifacts": 1,
                              "qnv8071/xacle-2027-data": 1}
artifact_roots = {
    p.parent.resolve() for p in KAGGLE_INPUT.rglob("moe8_gate")
    if p.is_dir() and (p.parent / "moescore_full8_store").is_dir()
}
if len(artifact_roots) != 1:
    raise FileNotFoundError("Expected one mounted Full8 artifact dataset with moe8_gate/ and "
                            f"moescore_full8_store/; found {sorted(map(str, artifact_roots))}")
ARTIFACT_ROOT = next(iter(artifact_roots))
ARTIFACTS = D02.resolve_precomputed_full8_artifacts(ARTIFACT_ROOT)
STORE_ROOT = ARTIFACTS["feature_store_root"]
metadata_dirs = {
    p.parent.resolve() for p in KAGGLE_INPUT.rglob("validation_average.csv")
    if "XACLE_Challenge_2027_development_dataset" in p.parts
    and not any("evaluation" in part.lower() for part in p.parts)
}
if len(metadata_dirs) > 1: raise ValueError(f"Ambiguous official metadata roots: {metadata_dirs}")
OFFICIAL_METADATA_ROOT = next(iter(metadata_dirs), None)
INPUT_PATHS = {"full8_artifacts": ARTIFACT_ROOT}
if OFFICIAL_METADATA_ROOT is not None: INPUT_PATHS["official_metadata"] = OFFICIAL_METADATA_ROOT
mark_stage("kaggle_input_discovery", "PASS", dataset_refs=DATASET_REFS,
           mounted_paths={k: str(v) for k, v in INPUT_PATHS.items()},
           official_metadata_attached=OFFICIAL_METADATA_ROOT is not None)

In [ ]:
# 5. Audit the gate bundle identity and SHA-256
RUN_MANIFEST = json.loads(ARTIFACTS["run_manifest"].read_text(encoding="utf-8"))
GATE_CONFIG = yaml.safe_load(ARTIFACTS["gate_config"].read_text(encoding="utf-8"))
CHECKPOINT_SHA256 = sha256_file(ARTIFACTS["gate_checkpoint"])
if RUN_MANIFEST.get("expert_profile") != PROFILE: raise ValueError("Run manifest is not full_8")
if RUN_MANIFEST.get("git_commit") != MOE8_SOURCE_COMMIT:
    raise ValueError("Run manifest source commit differs from checkpoint-compatible source")
if RUN_MANIFEST.get("gate_checkpoint_sha256") != CHECKPOINT_SHA256:
    raise ValueError("Checkpoint SHA-256 differs from run manifest")
if GATE_CONFIG.get("data", {}).get("profile") != PROFILE:
    raise ValueError("moe_full8.yaml does not declare full_8")
if not RUN_MANIFEST.get("source_code_sha256") or not RUN_MANIFEST.get("config_sha256"):
    raise ValueError("Run manifest is missing source/config fingerprints")
mark_stage("checkpoint_artifact_discovery", "PASS", checkpoint_sha256=CHECKPOINT_SHA256,
           source_commit=RUN_MANIFEST["git_commit"], checkpoint=str(ARTIFACTS["gate_checkpoint"]))

In [ ]:
# 6. Validate train, validation, and public-test feature stores
SPLITS = ("train", "validation", "test")
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}
DATASETS, FULL8_MANIFESTS = {}, {}
for split in SPLITS:
    # MoeTrainingDataset invokes validate_feature_store once per split, checking shard
    # hashes, finite arrays, dimensions, IDs, source identity, and within-split duplicates.
    dataset = MoeTrainingDataset(str(STORE_ROOT), split=split, require_labels=split != "test")
    DATASETS[split], FULL8_MANIFESTS[split] = dataset, dataset.manifest
    if len(dataset) != EXPECTED_COUNTS[split]:
        raise ValueError(f"{split}: expected {EXPECTED_COUNTS[split]}, found {len(dataset)}")
D02.validate_full8_manifest_contracts(FULL8_MANIFESTS, expected_counts=EXPECTED_COUNTS)
for split in ("train", "validation"):
    if FULL8_MANIFESTS[split]["store_identity_sha256"] != RUN_MANIFEST[f"{split}_store_identity_sha256"]:
        raise ValueError(f"{split}: store identity differs from run manifest")
mark_stage("full8_cache_validation", "PASS", sample_counts={s: len(DATASETS[s]) for s in SPLITS},
           store_identities={s: FULL8_MANIFESTS[s]["store_identity_sha256"] for s in SPLITS},
           cross_split_audio_overlap={"status": "SKIPPED",
               "reason": "Explicit D02 research configuration", "scope": list(SPLITS)})
write_json(OUTPUT_ROOT / "logs/preflight.json", {"stages": STAGES, "dataset_refs": DATASET_REFS,
    "mounted_paths": {k: str(v) for k, v in INPUT_PATHS.items()},
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "cross_split_audio_overlap": {"status": "SKIPPED",
        "reason": "Explicit D02 research configuration", "scope": list(SPLITS)}})

In [ ]:
# 7. Load the frozen gate using the source-compatible checkpoint manager
gate = GatingNetwork(FEATURE_WIDTH, len(EXPERT_ORDER), profile=PROFILE)
checkpoint_payload = MoeCheckpointManager().load_for_inference(
    ARTIFACTS["gate_checkpoint"], gating_network=gate,
    expected_expert_checkpoint_provenance=RUN_MANIFEST["expert_checkpoint_provenance"])
if checkpoint_payload.get("format_version") != 7 or checkpoint_payload.get("epoch") != 28:
    raise ValueError("Loaded gate schema or epoch differs from verified Full8 artifact")
if checkpoint_payload.get("source_code_sha256") != RUN_MANIFEST.get("source_code_sha256"):
    raise ValueError("Checkpoint source-file hashes differ from run manifest")
if checkpoint_payload.get("config_sha256") != RUN_MANIFEST.get("config_sha256"):
    raise ValueError("Checkpoint config fingerprint differs from run manifest")
if checkpoint_payload.get("expert_checkpoint_provenance") != RUN_MANIFEST.get("expert_checkpoint_provenance"):
    raise ValueError("Checkpoint expert provenance differs from run manifest")
D02.validate_checkpoint_store_compatibility(checkpoint_payload, FULL8_MANIFESTS)
gate.eval()
CHECKPOINT_META = {k: checkpoint_payload[k] for k in (
    "epoch", "format_version", "gate_architecture", "expert_checkpoint_provenance",
    "feature_store_identity", "source_code_sha256")}
mark_stage("checkpoint_store_compatibility", "PASS", epoch=CHECKPOINT_META["epoch"],
           schema=CHECKPOINT_META["format_version"], checkpoint="verified",
           feature_store="verified", optimizer_created=False)

In [ ]:
# 8. Frozen batched inference and one raw-score conversion
def gate_state_sha256(module):
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        digest.update(name.encode("utf-8"))
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

GATE_STATE_BEFORE = gate_state_sha256(gate)

def infer_split(split):
    loader = DataLoader(DATASETS[split], batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collate_moe_features)
    frames = []
    for batch in loader:
        with torch.inference_mode():
            weights = D02.infer_gate_weights(
                gate, batch["features"], batch_size=len(batch["features"]), device="cpu")
        scores = batch["scores"].numpy()
        labels = batch["labels_raw"].numpy() if "labels_raw" in batch else None
        frame = D02.build_per_sample_frame(batch["sample_ids"], batch["wave_file_names"],
                                           split, scores, weights, labels_raw=labels)
        expected_raw = (scores * weights.numpy()).sum(axis=1) * 5.0 + 5.0
        if not np.allclose(frame["predicted_score"], expected_raw, atol=1e-6, rtol=1e-6):
            raise RuntimeError(f"{split}: gate prediction/raw-score conversion mismatch")
        frame = frame.drop(columns=["oracle_diagnostic_only_score", "oracle_diagnostic_only_expert"], errors="ignore")
        frames.append(frame)
    result = pd.concat(frames, ignore_index=True)
    ids = [sid for shard in DATASETS[split].manifest["shards"] for sid in shard["sample_ids"]]
    if result["sample_id"].tolist() != ids or len(result) != EXPECTED_COUNTS[split]:
        raise ValueError(f"{split}: inferred sample IDs/count differ from verified store")
    return result

mark_stage("frozen_inference_ready", "PASS", device="cpu", batch_size=BATCH_SIZE,
           optimizer_created=False, training_performed=False)

In [ ]:
# 9. Frozen inference on official train
PER_SAMPLE = {}
if RUN_TRAIN_ANALYSIS:
    PER_SAMPLE["train"] = infer_split("train")
    mark_stage("train_inference", "PASS", samples=len(PER_SAMPLE["train"]))
else: mark_stage("train_inference", "SKIPPED", reason="Disabled by configuration")

In [ ]:
# 10. Frozen inference on official validation
if RUN_VALIDATION_ANALYSIS:
    PER_SAMPLE["validation"] = infer_split("validation")
    mark_stage("validation_inference", "PASS", samples=len(PER_SAMPLE["validation"]))
else: mark_stage("validation_inference", "SKIPPED", reason="Disabled by configuration")

In [ ]:
# 11. Frozen inference on official public test
if RUN_PUBLIC_TEST_ANALYSIS:
    PER_SAMPLE["test"] = infer_split("test")
    mark_stage("public_test_inference", "PASS", samples=len(PER_SAMPLE["test"]),
               labels="only from official metadata if available; evaluation-only")
else: mark_stage("public_test_inference", "SKIPPED", reason="Disabled by configuration")

In [ ]:
# 12. Score-bin/generator summaries and matched routing controls
metadata_identity = {}
for split, rows in list(PER_SAMPLE.items()):
    path = OFFICIAL_METADATA_ROOT / f"{split}_average.csv" if OFFICIAL_METADATA_ROOT else None
    if path is None or not path.is_file():
        mark_stage(f"{split}_metadata", "SKIPPED", reason="Official metadata not attached")
        continue
    meta = pd.read_csv(path)
    filename_key = next((k for k in ("wave_file_name", "wav_file_name", "filename", "file_name")
                         if k in meta.columns), None)
    model_key = next((k for k in ("model_id", "model_ID") if k in meta.columns), None)
    if filename_key is None or model_key is None:
        mark_stage(f"{split}_metadata", "SKIPPED", reason="Filename/model ID unavailable")
        continue
    cols = [filename_key, model_key] + (["model_name"] if "model_name" in meta.columns else [])
    selected = meta[cols].copy().rename(columns={model_key: "model_id"})
    score_key = "average_semantic_alignement_score"
    if split == "test" and score_key in meta.columns:
        selected["ground_truth_score"] = pd.to_numeric(meta[score_key], errors="raise")
        rows = rows.drop(columns=["ground_truth_score"])
    joined = D02.join_metadata_by_filename(rows, selected, filename_column=filename_key)
    if split == "test" and "ground_truth_score" in joined:
        y = joined["ground_truth_score"].to_numpy(dtype=float)
        if not np.isfinite(y).all() or np.any((y < 0) | (y > 10)):
            raise ValueError("Official public-test labels must be raw scores in [0,10]")
        joined["prediction_error"] = joined["predicted_score"] - y
        joined["absolute_prediction_error"] = np.abs(joined["prediction_error"])
        joined["ground_truth_score_bin"] = D02._score_bin_labels(y)
    PER_SAMPLE[split] = joined
    metadata_identity[split] = {"path": str(path), "sha256": sha256_file(path), "rows": len(meta)}
    mark_stage(f"{split}_metadata", "PASS", rows=len(joined),
               labels=("evaluation_only" if split == "test" and joined.ground_truth_score.notna().all()
                       else "store_labels_or_unavailable"))

expert_frames, bin_frames, generator_frames, control_frames = [], [], [], []
for split, rows in PER_SAMPLE.items():
    rows.to_csv(OUTPUT_ROOT / f"diagnostics/{split}_per_sample.csv", index=False)
    expert = D02.gate_weights_by_expert(rows); expert.insert(0, "split", split)
    expert_frames.append(expert)
    if rows.ground_truth_score.notna().all():
        bins = D02.score_bin_summary(rows); bins.insert(0, "split", split); bin_frames.append(bins)
        metrics = D02.compare_predictions(rows, bootstrap_samples=100, seed=2027)
    else:
        metrics = []
        for method, column in (("learned", "predicted_score"),
                               ("uniform", "uniform_prediction_score"),
                               ("hard_top1", "hard_top1_prediction_score")):
            pred = rows[column].to_numpy(dtype=float)
            metrics.append({"prediction_method": method, "diagnostic_only": True,
                "sample_count": len(rows), "srcc": None, "lcc": None, "kendall_tau_b": None,
                "mse": None, "rmse": None, "mae": None, "predicted_mean": float(pred.mean()),
                "predicted_std": float(pred.std()), "predicted_min": float(pred.min()),
                "predicted_max": float(pred.max()),
                "fraction_predicted_9_10": float(np.mean((pred >= 9) & (pred <= 10))),
                "fraction_predicted_0_2": float(np.mean((pred >= 0) & (pred < 2))),
                "fraction_predicted_7_8": float(np.mean((pred >= 7) & (pred < 8))),
                "fraction_predicted_8_9": float(np.mean((pred >= 8) & (pred < 9))),
                "label_metrics_status": "not_applicable_no_labels"})
        metrics = pd.DataFrame(metrics)
    metrics.insert(0, "split", split); control_frames.append(metrics)
    if "model_id" in rows:
        group = D02.generator_summary(rows); group.insert(0, "split", split)
        generator_frames.append(group)

gate_weights_by_expert = pd.concat(expert_frames, ignore_index=True)
control_comparison = pd.concat(control_frames, ignore_index=True)
gate_weights_by_expert.to_csv(OUTPUT_ROOT / "diagnostics/gate_weights_by_expert.csv", index=False)
control_comparison.to_csv(OUTPUT_ROOT / "diagnostics/control_comparison.csv", index=False)
if bin_frames:
    pd.concat(bin_frames, ignore_index=True).to_csv(
        OUTPUT_ROOT / "diagnostics/entropy_by_score_bin.csv", index=False)
if generator_frames:
    pd.concat(generator_frames, ignore_index=True).to_csv(
        OUTPUT_ROOT / "diagnostics/entropy_by_generator.csv", index=False)
else: mark_stage("generator_analysis", "SKIPPED", reason="No verified generator metadata")

In [ ]:
# 13. Figures
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
figdir = OUTPUT_ROOT / "figures"
colors = {"train": "#4477AA", "validation": "#EE6677", "test": "#228833"}

fig, ax = plt.subplots(figsize=(8, 5))
for split, rows in PER_SAMPLE.items():
    ax.hist(rows.normalized_entropy, bins=30, range=(0, 1), density=True, alpha=.42,
            label=f"{split} (n={len(rows)})", color=colors[split])
ax.set(xlabel="Normalized entropy", ylabel="Density", title="MOE8 gate entropy by split")
ax.legend(); fig.tight_layout(); fig.savefig(figdir / "entropy_by_split.png", dpi=180); plt.close(fig)

overall = gate_weights_by_expert.loc[gate_weights_by_expert.group_type == "overall"]
means = overall.pivot(index="expert", columns="split", values="mean_weight").reindex(EXPERT_ORDER)
fig, ax = plt.subplots(figsize=(9, 5))
means.plot(kind="bar", ax=ax, color=[colors[s] for s in means.columns])
ax.axhline(1/8, color="black", linestyle="--", label="Uniform 1/8")
ax.set(ylabel="Mean gate weight", xlabel="Expert", title="Mean gate weight by split")
ax.legend(); fig.tight_layout(); fig.savefig(figdir / "mean_gate_weights.png", dpi=180); plt.close(fig)

labeled = pd.concat([r for r in PER_SAMPLE.values() if r.ground_truth_score.notna().all()],
                    ignore_index=True)
if len(labeled):
    validation_rows = PER_SAMPLE["validation"]
    heat = validation_rows.groupby("ground_truth_score_bin")[
        [f"gate_weight_{i}" for i in range(1, 9)]].mean().reindex(D02.SCORE_BIN_LABELS)
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(heat.to_numpy().T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set(yticks=range(8), yticklabels=EXPERT_ORDER, xticks=range(len(heat)),
           xticklabels=heat.index, xlabel="Ground-truth score bin", title="Routing weight by score bin")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.colorbar(im, ax=ax, label="Mean gate weight"); fig.tight_layout()
    fig.savefig(figdir / "gate_weight_heatmap.png", dpi=180); plt.close(fig)

    if bin_frames:
        bins = pd.concat(bin_frames, ignore_index=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        for split, group in bins.groupby("split"):
            ax.plot(group.ground_truth_bin, group.mean_normalized_entropy, marker="o",
                    label=f"{split} (n={int(group.sample_count.sum())})", color=colors[split])
        ax.set(xlabel="Ground-truth score bin", ylabel="Normalized entropy",
               title="Gate entropy by score bin")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right"); ax.legend()
        fig.tight_layout(); fig.savefig(figdir / "entropy_by_score_bin.png", dpi=180); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    for split, rows in PER_SAMPLE.items():
        if rows.ground_truth_score.notna().all():
            ax.scatter(rows.ground_truth_score, rows.predicted_score, s=8, alpha=.22,
                       label=split, color=colors[split])
    ax.plot([0, 10], [0, 10], "k--", linewidth=1)
    ax.set(xlim=(0, 10), ylim=(0, 10), xlabel="Ground truth", ylabel="Prediction",
           title="Frozen MOE8 learned gate")
    ax.legend(); fig.tight_layout(); fig.savefig(figdir / "score_scatter.png", dpi=180); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    for split, rows in PER_SAMPLE.items():
        if rows.ground_truth_score.notna().all():
            ax.scatter(rows.normalized_entropy, rows.absolute_prediction_error, s=8, alpha=.22,
                       label=split, color=colors[split])
    ax.set(xlabel="Normalized entropy", ylabel="Absolute prediction error",
           title="Entropy and absolute error")
    ax.legend(); fig.tight_layout(); fig.savefig(figdir / "entropy_vs_error.png", dpi=180); plt.close(fig)

fig, axes = plt.subplots(1, len(PER_SAMPLE), figsize=(6 * len(PER_SAMPLE), 4), squeeze=False)
for ax, (split, rows) in zip(axes.flat, PER_SAMPLE.items()):
    for label, col in (("learned", "predicted_score"), ("uniform", "uniform_prediction_score"),
                       ("hard top-1", "hard_top1_prediction_score")):
        ax.hist(rows[col], bins=np.linspace(0, 10, 41), alpha=.4, label=label)
    ax.set(xlim=(0, 10), title=f"{split} (n={len(rows)})", xlabel="Raw XACLE score")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True)); ax.legend()
fig.tight_layout(); fig.savefig(figdir / "score_distributions.png", dpi=180); plt.close(fig)

if generator_frames:
    groups = pd.concat(generator_frames, ignore_index=True)
    groups = groups.loc[groups.split == "validation"].sort_values("mean_normalized_entropy")
    fig, ax = plt.subplots(figsize=(11, max(4, .28 * len(groups))))
    ax.barh(groups.model_id, groups.mean_normalized_entropy)
    ax.set(xlabel="Mean normalized entropy", ylabel="Generator",
           title="Validation entropy by generator")
    fig.tight_layout(); fig.savefig(figdir / "entropy_by_generator.png", dpi=180); plt.close(fig)
else: mark_stage("generator_figure", "SKIPPED", reason="Generator metadata unavailable")

high = PER_SAMPLE["validation"].loc[PER_SAMPLE["validation"].ground_truth_score >= 9]
if len(high):
    weight_means = high[[f"gate_weight_{i}" for i in range(1, 9)]].mean()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(EXPERT_ORDER, weight_means.to_numpy(), color="#4477AA")
    ax.set(ylabel="Mean gate weight", title=f"Score 9–10 routing (n={len(high)})")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    fig.tight_layout(); fig.savefig(figdir / "routing_by_score_bin.png", dpi=180); plt.close(fig)
mark_stage("figures", "PASS", files=sorted(p.name for p in figdir.glob("*.png")))

In [ ]:
# 14. Summarize observations and flag a possible G01 follow-up
validation = PER_SAMPLE["validation"]
entropy = validation.normalized_entropy
stds = {"learned": float(validation.predicted_score.std()),
        "uniform": float(validation.uniform_prediction_score.std()),
        "hard_top1": float(validation.hard_top1_prediction_score.std())}
g01_screen = bool(entropy.mean() >= .90 and
                  stds["learned"] < max(stds["uniform"], stds["hard_top1"]))
summary = {
    "experiment_id": EXPERIMENT_ID, "profile": PROFILE,
    "observations": {
        "mean_normalized_entropy": {s: float(r.normalized_entropy.mean()) for s, r in PER_SAMPLE.items()},
        "fraction_entropy_ge_0_90": {s: float((r.normalized_entropy >= .90).mean()) for s, r in PER_SAMPLE.items()},
        "fraction_entropy_ge_0_95": {s: float((r.normalized_entropy >= .95).mean()) for s, r in PER_SAMPLE.items()},
        "mean_gate_weight": {s: {f"expert{i}": float(r[f"gate_weight_{i}"].mean())
                                 for i in range(1, 9)} for s, r in PER_SAMPLE.items()},
        "validation_prediction_std": stds,
        "control_metrics": control_comparison.astype(object).where(pd.notna(control_comparison), None).to_dict(orient="records"),
    },
    "g01_follow_up_screen": {"suggested": g01_screen,
        "rule": "Validation mean normalized entropy >= 0.90 and learned output std below a control.",
        "interpretation": "Descriptive prioritization only, not evidence entropy regularization will help."},
    "limitations": {
        "causality": "D02 cannot establish that entropy causes score compression or prediction errors.",
        "public_test_labels": "If present, official labels are evaluation-only and never used for training, fitting, tuning, or checkpoint selection.",
        "cross_split_audio_overlap": {"status": "SKIPPED",
            "reason": "Explicit D02 research configuration", "scope": list(SPLITS),
            "split_independence_established": False,
            "risk": "Undetected duplicate audio may make evaluation estimates optimistic."},
    },
}
write_json(OUTPUT_ROOT / "reports/d02_summary.json", summary)
report = ["# D02-MOE8 gate entropy diagnostics", "", "## Validation observations", "",
    f"- Samples: {len(validation)}", f"- Mean normalized entropy: {entropy.mean():.4f}",
    f"- Fraction with entropy >= 0.90: {(entropy >= .90).mean():.4f}",
    f"- Fraction with entropy >= 0.95: {(entropy >= .95).mean():.4f}",
    f"- Learned/uniform/hard top-1 prediction standard deviations: {stds}",
    f"- G01 follow-up screen suggested: {g01_screen}", "",
    "A wider output range alone does not show better ranking or accuracy. D02 is descriptive, not causal, and does not start G01.", "",
    "## Matched validation controls", "",
    control_comparison.loc[control_comparison.split == "validation"].to_string(index=False), "",
    "## Limits", "",
    "Cross-split WAV overlap checks were skipped; split independence is not established. Public-test labels, if present, were evaluation-only.",
    "No private evaluation data was attached or accessed."]
(OUTPUT_ROOT / "reports/d02_report.md").write_text("\n".join(report) + "\n", encoding="utf-8")
mark_stage("summary", "PASS", g01_follow_up_screen=g01_screen, causal_claim=False)

In [ ]:
# 15. Export D02 artifacts and provenance manifest
GATE_STATE_AFTER = gate_state_sha256(gate)
if GATE_STATE_AFTER != GATE_STATE_BEFORE: raise RuntimeError("Gate parameters changed during inference")
if gate.training or any(p.grad is not None for p in gate.parameters()):
    raise RuntimeError("Gate is in training mode or has gradients")
mark_stage("frozen_gate_integrity", "PASS", state_sha256=GATE_STATE_AFTER,
           optimizer_created=False, training_performed=False)

manifest_path = OUTPUT_ROOT / "reports/artifact_manifest.json"
paths = sorted(str(p.relative_to(OUTPUT_ROOT)) for p in OUTPUT_ROOT.rglob("*")
               if p.is_file() and p != manifest_path)
artifact_manifest = {
    "experiment_id": EXPERIMENT_ID, "profile": PROFILE,
    "source_commit": MOE8_SOURCE_COMMIT, "d02_analysis_commit": D02_HELPER_COMMIT,
    "checkpoint_sha256": CHECKPOINT_SHA256, "checkpoint_epoch": CHECKPOINT_META["epoch"],
    "checkpoint_format_version": CHECKPOINT_META["format_version"],
    "expert_order": list(EXPERT_ORDER), "expert_count": 8, "feature_width": FEATURE_WIDTH,
    "splits": {s: len(r) for s, r in PER_SAMPLE.items()},
    "feature_store_identity": {s: {
        "store_identity_sha256": FULL8_MANIFESTS[s]["store_identity_sha256"],
        "source_manifest_sha256": FULL8_MANIFESTS[s]["source_manifest_sha256"],
    } for s in PER_SAMPLE},
    "dataset_identity": {"refs": DATASET_REFS, "versions_at_upload": DATASET_VERSIONS_AT_UPLOAD,
                         "mounted_paths": {k: str(v) for k, v in INPUT_PATHS.items()},
                         "official_metadata": metadata_identity},
    "cache_provenance": "verified_by_Full8_manifests_and_checkpoint_manager",
    "checkpoint_provenance": "verified_by_MoeCheckpointManager_and_run_manifest",
    "cross_split_audio_overlap": {"status": "SKIPPED",
        "reason": "Explicit D02 research configuration", "scope": list(SPLITS),
        "split_independence_established": False},
    "training_performed": False, "experts_loaded_for_extraction": False,
    "private_evaluation_dataset_accessed": False, "execution_device": "cpu",
    "analysis_parameters": {"batch_size": BATCH_SIZE, "entropy_thresholds": [.90, .95],
                            "bootstrap_samples": 100, "random_seed": 2027},
    "artifact_paths": paths + ["reports/artifact_manifest.json"],
    "execution_status": "completed", "stages": STAGES,
}
write_json(manifest_path, artifact_manifest)
write_json(OUTPUT_ROOT / "logs/preflight.json", {
    "stages": STAGES, "dataset_refs": DATASET_REFS,
    "mounted_paths": {k: str(v) for k, v in INPUT_PATHS.items()},
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "feature_store_identity": artifact_manifest["feature_store_identity"],
    "cross_split_audio_overlap": artifact_manifest["cross_split_audio_overlap"],
})
print(f"D02 outputs saved under {OUTPUT_ROOT}")
print("\n".join("- " + p for p in artifact_manifest["artifact_paths"]))